# **Question 6: Advanced - Object-Oriented Design (Interfaces)**

In a mature MLOps platform, you often have a **Model Registry**. Your system needs to support models from different frameworks: Scikit-Learn, PyTorch, TensorFlow, and XGBoost.

You want to write a piece of infrastructure code (a "Runner") that takes a model and generates predictions. However, Scikit-Learn uses `.predict()`, PyTorch uses `model(x)`, and TensorFlow might use `.call()`.

You want to enforce a **Standard Interface** so that any model added to your system *must* implement a specific method called `infer(data)`, otherwise the code shouldn't even run.

**The Question:**
1.  Which Python module (built-in) allows you to create **Abstract Base Classes**?
2.  How do you define a class that *cannot* be instantiated itself but forces its child classes to implement specific methods?
3.  What happens if a Junior Engineer inherits from your base class but forgets to write the `infer` method?

---

## Background (Why This Matters in MLOps)

A mature MLOps platform supports models from multiple frameworks:
- **Scikit-Learn** → uses `.predict()`
- **PyTorch** → uses `model(x)` (callable)
- **TensorFlow** → uses `.call()`
- **XGBoost** → uses `.predict()`

Without a standard interface, your infrastructure code becomes a mess of `if/elif` chains — hard to test, hard to extend, impossible to scale.

The solution: enforce a **contract** using Abstract Base Classes. Any model entering the system *must* implement `infer(data)` — or it fails immediately, before it ever reaches production.

---

## Q1 — Which Built-in Python Module Creates Abstract Base Classes?

**Answer: `abc`** (Abstract Base Classes)

```python
from abc import ABC, abstractmethod
```

Two things to import:
- `ABC` — the base class your abstract class inherits from
- `abstractmethod` — the decorator that marks methods as required

---

## Q2 — How Do You Define a Class That Cannot Be Instantiated and Forces Child Classes to Implement Methods?

### The Pattern

```python
from abc import ABC, abstractmethod

class ModelBase(ABC):

    @abstractmethod
    def infer(self, data):
        """Every model MUST implement this."""
        pass

    @abstractmethod
    def load(self):
        """Every model MUST implement this."""
        pass
```

### Key Points
- `ModelBase` **inherits from `ABC`** — this makes it abstract
- `@abstractmethod` marks methods that **subclasses must implement**
- `ModelBase` itself **cannot be instantiated** — it's a contract, not an implementation
- If you try `ModelBase()` → Python raises `TypeError` immediately

### Concrete Implementations (The Plug-ins)

```python
class SklearnModel(ModelBase):
    def load(self):
        import joblib
        self.model = joblib.load("model.pkl")

    def infer(self, data):
        return self.model.predict(data)


class TorchModel(ModelBase):
    def load(self):
        import torch
        self.model = torch.load("model.pt")
        self.model.eval()

    def infer(self, data):
        import torch
        with torch.no_grad():
            return self.model(data).numpy()


class TensorFlowModel(ModelBase):
    def load(self):
        import tensorflow as tf
        self.model = tf.saved_model.load("model_dir")

    def infer(self, data):
        return self.model(data).numpy()
```

---

## Q3 — What Happens If a Junior Engineer Forgets to Implement `infer`? (The Critical Detail)

### The Answer (Exact Interview Phrasing)

> "Python raises a `TypeError` **at the moment of instantiation** — not when `infer` is eventually called. The invalid class is blocked from being created at all."

### Demonstration

```python
# Junior engineer writes this — forgets infer()
class BrokenModel(ModelBase):
    def load(self):
        pass
    # FORGOT to implement infer()

# This line FAILS immediately with TypeError
model = BrokenModel()
# TypeError: Can't instantiate abstract class BrokenModel
# with abstract method infer
```

### Why the Timing Matters (This is the "Hire" Factor)

| Approach | When the bug is caught | Risk |
|---|---|---|
| No ABC (plain class) | When `infer()` is called at runtime | A real user hits the API and gets a crash |
| With `@abstractmethod` | **At instantiation** — app startup or test run | Bug caught before any user is affected |

> In production, "fail fast at startup" is infinitely better than "fail at runtime under load."

---

## Internal Flow — What Python Actually Does

```python
# Python checks at instantiation:
# "Does this class implement ALL @abstractmethod methods?"

# If YES → object is created normally
model = SklearnModel()   # ✅ works fine

# If NO → TypeError is raised IMMEDIATELY
model = BrokenModel()    # ❌ TypeError at this line

# The base class itself is also blocked
model = ModelBase()      # ❌ TypeError — abstract class
```

Python tracks unimplemented abstract methods in `__abstractmethods__` — a frozenset on the class. If it's non-empty, instantiation is blocked.

```python
print(BrokenModel.__abstractmethods__)
# frozenset({'infer'})
```

---

## The Caller Code — Most Important Part

The power of this pattern is that **caller code never changes**, regardless of which model is injected.

```python
def serve_prediction(model: ModelBase, input_data):
    """
    This function does NOT care:
    - which ML framework is used
    - how the model was trained
    - where it is running
    It only knows: model.load() and model.infer() exist.
    """
    model.load()
    return model.infer(input_data)
```

### Configuration-Driven Model Selection

```yaml
# config.yaml
model:
  type: sklearn
```

```python
def get_model(config) -> ModelBase:
    registry = {
        "sklearn": SklearnModel,
        "torch":   TorchModel,
        "tf":      TensorFlowModel,
    }
    model_cls = registry.get(config["model"]["type"])
    if not model_cls:
        raise ValueError(f"Unknown model type: {config['model']['type']}")
    return model_cls()

# End-to-end flow
config = load_config()
model  = get_model(config)
result = serve_prediction(model, input_data)
# Caller code NEVER changes — only the config does
```

---

## Real Production Example — FastAPI

```python
from fastapi import FastAPI, Depends

app = FastAPI()

def get_model_dependency() -> ModelBase:
    config = load_config()
    return get_model(config)

@app.post("/predict")
def predict_endpoint(
    payload: dict,
    model: ModelBase = Depends(get_model_dependency)
):
    return {"prediction": model.infer(payload)}

# FastAPI injects the correct model implementation
# The endpoint code is 100% framework-agnostic
```

---

## Testing Becomes Trivial (Huge MLOps Win)

```python
class DummyModel(ModelBase):
    """Fake model for unit tests — zero dependencies."""
    def load(self):
        pass

    def infer(self, data):
        return {"result": "test_prediction"}

# Test the caller code with no real ML framework needed
def test_serve_prediction():
    model = DummyModel()
    result = serve_prediction(model, {"feature": 1.0})
    assert result == {"result": "test_prediction"}
```

> Because caller code depends on `ModelBase`, not on `SklearnModel`, you can swap in a `DummyModel` with zero friction.

---

## Without vs With Abstraction

### Without ABC (Bad)
```python
# Every new framework = more if/elif
def serve_prediction(model_type, model, data):
    if model_type == "sklearn":
        return model.predict(data)
    elif model_type == "torch":
        with torch.no_grad():
            return model(data)
    elif model_type == "tf":
        return model.call(data)
    # What happens when XGBoost is added? Edit this function again.
```
- Code duplication
- Hard to test (must test every branch)
- Hard to extend (touch existing code every time)
- Violates Open/Closed Principle

### With ABC (Good)
```python
def serve_prediction(model: ModelBase, data):
    return model.infer(data)
# Adding XGBoost = write a new class, touch nothing else
```
- Clean single-responsibility
- Testable with a DummyModel
- Extensible without touching existing code
- Follows Open/Closed Principle

---

## Senior Engineer Answer (Full Interview Script)

> "I would use Python's built-in `abc` module. I'd define a base class called `ModelBase` that inherits from `ABC`, and mark the `infer` method with the `@abstractmethod` decorator.
>
> The key behavior is: if any subclass fails to implement `infer`, Python raises a `TypeError` **at the moment of instantiation** — not when `infer` is eventually called at runtime. This is critical in MLOps because it means a misconfigured model is caught at application startup or during tests, before any user ever hits the API.
>
> In practice, the caller code — whether it's a FastAPI endpoint or a batch runner — only depends on `ModelBase`, never on `SklearnModel` or `TorchModel`. This makes the system easy to extend, easy to test with a `DummyModel`, and configuration-driven via a simple registry pattern."

---

## Mental Model — Lock This In

```
Caller code talks to the INTERFACE, not the IMPLEMENTATION.

Caller asks:  "Can you infer?"
NOT:          "Are you Sklearn or Torch?"
```

| Layer | What it knows |
|---|---|
| Caller (`serve_prediction`) | Only `ModelBase` — the contract |
| Factory (`get_model`) | Maps config → concrete class |
| Implementation (`SklearnModel`) | The framework details |

---

## One-Line Revision Summary

> Import `ABC` and `abstractmethod` from the `abc` module → inherit from `ABC` → decorate required methods with `@abstractmethod` → Python raises `TypeError` **at instantiation** (not at call time) if any abstract method is missing — blocking invalid classes before they ever reach runtime.

---

## Interview Delivery Tips

1. **Nail the timing:** The #1 differentiator — say "at instantiation, not at call time." Most candidates say "it raises an error when you call the method" which is wrong.
2. **Use `__abstractmethods__`** as a bonus detail — shows you understand the internals.
3. **Mention the testing win:** `DummyModel` for unit tests shows you think in systems, not just syntax.
4. **Connect to Open/Closed Principle:** "Adding XGBoost means writing a new class, not editing existing code" — this signals design maturity.
5. **FastAPI / config registry** — shows you've thought about how this works end-to-end in production.